In [2]:
import os
import csv
import pandas as pd
import xgboost as xgb
import m2cgen as m2c
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [15]:
def preprocess_data(data_dir, output_csv):
    # 1. Define the correct subfolder
    source_dir = os.path.join(data_dir, "Simplified")
    
    print(f"Parsing files from: {source_dir}")
    
    with open(output_csv, mode='w', newline='') as f:
        writer = csv.writer(f)
        header = ['gesture_type', 'label']
        joints = ['L_Sh', 'L_El', 'L_Wr', 'R_Sh', 'R_El', 'R_Wr']
        for j in joints: header.extend([f'{j}_x', f'{j}_y', f'{j}_z'])
        writer.writerow(header)

        # 2. Iterate through files in the 'Simplified' folder
        for filename in os.listdir(source_dir):
            if filename.endswith(".txt"):
                parts = filename.split('_')
                
                # Safety check: ensure filename has enough parts
                if len(parts) < 5: continue
                
                gesture = parts[2]
                label = 1 if parts[4] == '1' else 0 
                
                # 3. FIX: Open using source_dir (the subfolder), not data_dir
                file_path = os.path.join(source_dir, filename)
                
                with open(file_path, 'r') as file:
                    for line in file:
                        raw = line.replace(',', ' ').split()
                        if len(raw) < 75: continue
                        
                        idx = [12, 15, 18, 24, 27, 30] 
                        row = [gesture, label]
                        for i in idx:
                            row.extend([float(raw[i]), float(raw[i+1]), float(raw[i+2])])
                        writer.writerow(row)
    print("Preprocessing complete.")

# Execution
preprocess_data("./SkeletonData", "data.csv")


Parsing files from: ./SkeletonData/Simplified
Preprocessing complete.


In [18]:
from sklearn.model_selection import train_test_split
def train_and_export(csv_file):
    df = pd.read_csv(csv_file)
    
    # 1. Find which gestures are actually available
    available_gestures = df['gesture_type'].unique()
    print(f"Gestures available in dataset: {available_gestures}")
    
    if len(available_gestures) == 0:
        print("Error: No data found in CSV.")
        return

    # 2. Pick the first available gesture automatically
    target_gesture = available_gestures[0]
    print(f"Training model for Gesture ID: {target_gesture}")
    
    df = df[df['gesture_type'] == target_gesture]
    
    # 3. Safety check: Do we have enough data to split?
    if len(df) < 10:
        print(f"Error: Not enough data for gesture {target_gesture} (Found {len(df)} rows). Need at least 10.")
        return
    
    X = df.drop(columns=['gesture_type', 'label'])
    y = df['label']
    
    # 4. Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 5. Train
    model = xgb.XGBClassifier(
        n_estimators=100, 
        max_depth=5, 
        base_score=0.5, # <--- Add this line to fix the TypeError
        eval_metric='logloss' # <--- Good practice to avoid warnings
    )
    model.fit(X_train, y_train)
    
    accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f"Model Accuracy for Gesture {target_gesture}: {accuracy*100:.2f}%")

    # 6. Export
    js_code = m2c.export_to_javascript(model)
    with open("XGBoost_Engine.js", "w") as f:
        f.write("export function evaluatePoseForm(features) {\n")
        f.write(js_code)
        f.write("\n}")
    print("Model exported to XGBoost_Engine.js!")

# Run it
train_and_export("data.csv")

Gestures available in dataset: [3 7 0 2 6 5 1 4 8]
Training model for Gesture ID: 3
Model Accuracy for Gesture 3: 99.92%
Model exported to XGBoost_Engine.js!


In [10]:
import os

# Peek at the first 5 filenames in your directory
data_dir = "./SkeletonData"
files = [f for f in os.listdir(data_dir) if f.endswith(".txt")]

print("Checking first 5 filenames:")
for f in files[:5]:
    parts = f.split('_')
    print(f"File: {f} | Parts: {parts}")


Checking first 5 filenames:


In [12]:
import os
print("Current Working Directory:", os.getcwd())
print("Contents of current directory:", os.listdir())

Current Working Directory: /Users/chanpanha/Desktop/RehabAI
Contents of current directory: ['.DS_Store', 'requirements.txt', 'train_model.ipynb', 'dataset', 'SkeletonData', 'data.csv', '.venv']


In [11]:
df = pd.read_csv("data.csv")
print("These are the unique values currently in 'gesture_type':")
print(df['gesture_type'].unique())

These are the unique values currently in 'gesture_type':
[]


In [13]:
# Peek at the filenames inside your SkeletonData folder
files = os.listdir("./SkeletonData")
print("First 5 filenames:", files[:5])

First 5 filenames: ['Simplified', '.DS_Store', 'RawData']
